In [1]:
from tqdm import tqdm
import os
from os import listdir
import time
from random import randint
from os.path import isfile, join
 
import gc 
import numpy as np
from scipy import stats
import pandas as pd
import pickle as pkl

from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.model_selection import KFold

import nibabel as nib
import pydicom as pdm
import nilearn as nl
import nilearn.plotting as nlplt
import h5py

from skimage import feature

import matplotlib.pyplot as plt
from matplotlib import cm
import matplotlib.animation as anim
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec

import seaborn as sns
import imageio
from skimage.transform import resize
from skimage.util import montage

# from IPython.display import Image as show_gif
# from IPython.display import clear_output
# from IPython.display import YouTubeVideo

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F

from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.nn import MSELoss

# !pip install opencv-python==4.6.0.66
# !pip install -U albumentations --no-binary qudida,albumentations
import albumentations as A
# from albumentations.pytorch import ToTensor, ToTensorV2


from albumentations import Compose, HorizontalFlip
# from albumentations.pytorch import ToTensor, ToTensorV2 


from tqdm import tqdm_notebook as tqdm
from multiprocessing import Pool

import warnings
warnings.simplefilter("ignore")

# Definations and descriptions of GLCM Texture Features

## 1. Contrast
- **Description**: An image divided in half, one side very bright and the other very dark.
- **Formula**: \(\sum_{i,j=0}^{levels-1} (i - j)^2 \times M(i, j)\)
- **Definition**: Measures the local variations in the GLCM. High contrast values indicate large variations in intensity levels.
- **Significance**: Represents high contrast, indicating large variations in intensity levels.

## 2. Dissimilarity
- **Description**: An image with gradual texture changes.
- **Formula**: \(\sum_{i,j=0}^{levels-1} |i - j| \times M(i, j)\)
- **Definition**: Similar to contrast but gives less weight to intensity differences that are further apart.
- **Significance**: Illustrates the concept of dissimilarity, where changes in texture are less abrupt.

## 3. Homogeneity
- **Description**: A smooth or blurred image.
- **Formula**: \(\sum_{i,j=0}^{levels-1} \frac{M(i, j)}{1 + (i - j)^2}\)
- **Definition**: Measures the closeness of the distribution of elements in the GLCM to the GLCM diagonal.
- **Significance**: Indicates a high degree of homogeneity, where intensity transitions are subtle.

## 4. Energy (ASM)
- **Description**: An image with a constant or very repetitive texture.
- **Formula**: \(\sum_{i,j=0}^{levels-1} M(i, j)^2\)
- **Definition**: Provides the sum of squared elements in the GLCM, a measure of textural uniformity.
- **Significance**: Demonstrates high energy or Angular Second Moment, indicative of textural uniformity.

## 5. Entropy
- **Description**: A highly detailed and complex image, like a forest scene.
- **Formula**: \(-\sum_{i,j=0}^{levels-1} M(i, j) \log(M(i, j))\)
- **Definition**: Measures the randomness in the image texture. Higher values indicate more complexity or randomness.
- **Significance**: Exemplifies high entropy, reflecting complexity or randomness in the image texture.


# Function to Calculate Volume of a Tumor based on mask file

In [46]:
def preprocess_mask_labels(mask):
    # whole tumour
    mask_WT = mask.copy()
    mask_WT[mask_WT == 1] = 1
    mask_WT[mask_WT == 2] = 1
    mask_WT[mask_WT == 3] = 1
    # include all tumours 

    # NCR / NET - LABEL 1
    mask_TC = mask.copy()
    mask_TC[mask_TC == 1] = 1
    mask_TC[mask_TC == 2] = 0
    mask_TC[mask_TC == 3] = 1
    # exclude 2 / 4 labelled tumour 

    # ET - LABEL 4 
    mask_ET = mask.copy()
    mask_ET[mask_ET == 1] = 0
    mask_ET[mask_ET == 2] = 0
    mask_ET[mask_ET == 3] = 1
    # exclude 2 / 1 labelled tumour 

    mask = np.stack([mask_WT, mask_TC, mask_ET, mask_ET])
    
    return mask 

def read_MRI(dataset, patient_id):
    if dataset == 'Brats2020':
        baseloc = '../input/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData/'
        pefix = 'BraTS20_Training_' + patient_id + '/' + 'BraTS20_Training_' + patient_id
        suffixs = ['_flair.nii','_t2.nii', '_t1.nii', '_t1ce.nii', '_seg.nii']
    elif dataset == 'Brats2023':
        baseloc = '../input/Brats2023/ASNR-MICCAI-BraTS2023-GLI-Challenge-TrainingData/'
        pefix = 'BraTS-GLI-' + patient_id + '/' + 'BraTS-GLI-' + patient_id
        suffixs = ['-t2f.nii.gz','-t2w.nii.gz', '-t1n.nii.gz', '-t1c.nii.gz', '-seg.nii.gz']

    flair_filename = baseloc + pefix + suffixs[0]
    flair_img_f = nib.load(flair_filename)
    flair_img = np.asarray(flair_img_f.dataobj)

    t2_filename = baseloc + pefix + suffixs[1]
    t2_img_f = nib.load(t2_filename)
    t2_img = np.asarray(t2_img_f.dataobj)

    t1_filename = baseloc + pefix + suffixs[2]
    t1_img_f = nib.load(t1_filename)
    t1_img = np.asarray(t1_img_f.dataobj)

    t1ce_filename = baseloc + pefix + suffixs[3]
    t1ce_img_f = nib.load(t1ce_filename)
    t1ce_img = np.asarray(t1ce_img_f.dataobj)
    
    mask_filename = baseloc + pefix + suffixs[4]
    mask_img_f = nib.load(mask_filename)
    mask_img = np.asarray(mask_img_f.dataobj)
    
    return flair_img, t2_img, t1_img, t1ce_img, mask_img 

def calculate_bounding_box(data):
    """
    Calculate the bounding box for the region of interest in a 3D MRI mask file.

    :param data: The MRI image mask.
    :return: roi_start, roi_end coordinates.
    """

    # Find the indices where the tumor is present
    indices = np.array(np.where(data == 1))

    # Calculate the bounding box
    roi_start = np.min(indices, axis=1)
    roi_end = np.max(indices, axis=1) + 1  # Add 1 to include the end index

    return tuple(roi_start), tuple(roi_end)


def crop_mri_mask(data, roi_start, roi_end):
    """
    Crop a region of interest from a 3D MRI mask file.

    :param data: The MRI image mask.
    :param roi_start: The start coordinates (x, y, z) of the ROI.
    :param roi_end: The end coordinates (x, y, z) of the ROI.
    :return: Cropped MRI data.
    """
    # Crop the data
    cropped_data = data[roi_start[0]:roi_end[0], roi_start[1]:roi_end[1], roi_start[2]:roi_end[2]]

    return cropped_data

def saperate_tumor_and_background(sample_img1, sample_img2, sample_img3, sample_img4, mask):
    masks = preprocess_mask_labels(mask)
    mask_WT, mask_TC, mask_ET = masks[0], masks[1], masks[2]
    
    tumor_only_sample_img1 = np.where(mask_WT == 1, sample_img1, 0)
    tumor_blacked_sample_img1 = np.where(mask_WT == 1, 0, sample_img1)

    tumor_only_sample_img2 = np.where(mask_WT == 1, sample_img2, 0)
    tumor_blacked_sample_img2 = np.where(mask_WT == 1, 0, sample_img2)

    tumor_only_sample_img3 = np.where(mask_WT == 1, sample_img3, 0)
    tumor_blacked_sample_img3 = np.where(mask_WT == 1, 0, sample_img3)

    tumor_only_sample_img4 = np.where(mask_WT == 1, sample_img4, 0)
    tumor_blacked_sample_img4 = np.where(mask_WT == 1, 0, sample_img4)
    return {'tumor_only': [tumor_only_sample_img1, 
                           tumor_only_sample_img2, 
                           tumor_only_sample_img3, 
                           tumor_only_sample_img4], 
            'tumor_blacked': [tumor_blacked_sample_img1, 
                              tumor_blacked_sample_img2, 
                              tumor_blacked_sample_img3, 
                              tumor_blacked_sample_img4]}

def calculate_overall_texture(dataset, patient_id):
    flair_img, t2_img, t1_img, t1ce_img, mask_img   = read_MRI(dataset, patient_id)
    
    stacked_image = {'t1':t1_img, 
                    't1ce':t1ce_img, 
                    't2':t2_img, 
                    'flair':flair_img}
    
    image_textures = {}
    for modality in stacked_image.keys():
        image = stacked_image[modality]
    
        # Convert to uint8 for texture analysis
        sample_img_uint8 = (image / image.max() * 255).astype(np.uint8)
        contrast = {}
        dissimilarity = {}
        homogeneity = {}
        energy = {}
        correlation = {}
        ASM = {}
        for _slice in range(image.shape[2]):
            # Compute GLCM and texture properties for a slice
            slice_sample_img = sample_img_uint8[:, :, _slice]
            glcm = feature.graycomatrix(slice_sample_img, 
                                        distances=np.linspace(1, 10, 10),
                                        angles=[0, 3.14/4, 3.14/2,  (3*3.14)/4], 
                                        symmetric=True, 
                                        normed=True)


            # Compute texture properties
            contrast[_slice] = feature.graycoprops(glcm, 'contrast')
            dissimilarity[_slice] = feature.graycoprops(glcm, 'dissimilarity')
            homogeneity[_slice] = feature.graycoprops(glcm, 'homogeneity')
            energy[_slice] = feature.graycoprops(glcm, 'energy')
            correlation[_slice] = feature.graycoprops(glcm, 'correlation')
            ASM[_slice] = feature.graycoprops(glcm, 'ASM')
        
        image_textures[modality] = {'contrast': contrast, 
                                    'dissimilarity': dissimilarity, 
                                    'homogeneity': homogeneity, 
                                    'energy': energy, 
                                    'correlation':correlation, 
                                    'ASM':ASM}
    
    return image_textures

def extract_patches(data, mask, patch_size=10, num_samples=10, intensity_threshold=0, max_iterations=1000):
    """ Extract small patch samples from tumor and non-tumor regions in 2D images across multiple channels """

    # Function to extract a patch from all channels centered at a specific coordinate
    def extract_multichannel_patch(center, patch_type='tumor'):
        start = np.maximum(center - patch_size // 2, 0)
        end = np.minimum(center + patch_size // 2 + 1, data.shape[1:])

        patches = [channel[start[0]:end[0], start[1]:end[1]] for channel in data]
        mask_patch = mask[start[0]:end[0], start[1]:end[1]]

        # Ensure the patch is entirely within the desired region (tumor or non-tumor)
        if patch_type == 'tumor' and np.all(mask_patch == 1):
            return patches
        elif patch_type == 'non_tumor' and np.all((mask_patch == 0) & (patches[0] > intensity_threshold)):
            return patches
        else:
            return None

    # Extract tumor and non-tumor patches
    tumor_patches = []
    non_tumor_patches = []
    
    # Coordinates where the mask is positive (tumor) and negative (non-tumor)
    tumor_coords = np.argwhere(mask == 1)
    non_tumor_coords = np.argwhere((mask == 0) & (data[0] > intensity_threshold))
    
    if (len(tumor_coords) <= (patch_size**2)):
        return None, None

    iterations = 0
    while len(tumor_patches) < num_samples and iterations < max_iterations:
        coord = tumor_coords[np.random.choice(len(tumor_coords))]
        patch = extract_multichannel_patch(coord, patch_type='tumor')
        if patch is not None:
            tumor_patches.append(patch)
        iterations += 1
    
    if len(tumor_patches) <= num_samples/4:
        return None, None

    iterations = 0
    while len(non_tumor_patches) < num_samples and iterations < max_iterations:
        coord = non_tumor_coords[np.random.choice(len(non_tumor_coords))]
        patch = extract_multichannel_patch(coord, patch_type='non_tumor')
        if patch is not None:
            non_tumor_patches.append(patch)
        iterations += 1
    
    if len(non_tumor_patches) <= num_samples/4:
        return None, None

    return np.array(tumor_patches), np.array(non_tumor_patches)



def calculate_textures(patches, modalities):
    image_textures = {}
    for modality in modalities.keys():
        contrast = {}
        dissimilarity = {}
        homogeneity = {}
        energy = {}
        correlation = {}
        ASM = {}
        modality_num = modalities[modality]
        patch_num = 0
        for patch in patches[:,modality_num,:,:]:
            normalized_patch = (patch - np.min(patch)) / (np.max(patch) - np.min(patch))
            normalized_patch = (normalized_patch * 255).astype(np.uint8)
             # Compute GLCM and texture properties for a patch
            glcm = feature.graycomatrix(normalized_patch, 
                                        distances=np.linspace(1, 3, 3),
                                        angles=[0, 3.14/4, 3.14/2,  (3*3.14)/4], 
                                        symmetric=True, 
                                        normed=True)


            # Compute texture properties
            contrast[patch_num] = feature.graycoprops(glcm, 'contrast')
            dissimilarity[patch_num] = feature.graycoprops(glcm, 'dissimilarity')
            homogeneity[patch_num] = feature.graycoprops(glcm, 'homogeneity')
            energy[patch_num] = feature.graycoprops(glcm, 'energy')
            correlation[patch_num] = feature.graycoprops(glcm, 'correlation')
            ASM[patch_num] = feature.graycoprops(glcm, 'ASM')
            patch_num += 1
        
        
        image_textures[modality] = {'contrast': contrast, 
                                    'dissimilarity': dissimilarity, 
                                    'homogeneity': homogeneity, 
                                    'energy': energy, 
                                    'correlation':correlation, 
                                    'ASM':ASM}
        print(energy)
    return image_textures
    

def calculate_texture_from_patches(dataset, patient_id):
    flair_img, t2_img, t1_img, t1ce_img, mask_img   = read_MRI(dataset, patient_id)
    
    modalities = {'t1':0, 
                    't1ce':1, 
                    't2':2, 
                    'flair':3}

    tumor_image_textures = {}
    non_tumor_image_textures = {}
    all_image_textures = {}
    
    stacked_image = np.stack([t1_img, t1ce_img, t2_img, flair_img])
    mask_image = mask_img
    
    
    for _slice in tqdm(range(stacked_image.shape[3])):
        # Extract cubes
        mri_data = stacked_image[:,:,:,_slice]
        mask_data = mask_image[:,:,_slice]
        tumor_patches, non_tumor_patches = extract_patches(mri_data, mask_data, num_samples=100)
        
        tumor_image_texture = {'t1':{}, 't1ce':{}, 't2':{}, 'flair':{}}
        non_tumor_image_texture = {'t1':{}, 't1ce':{}, 't2':{}, 'flair':{}}
        
        if tumor_patches is not None:
            tumor_image_texture = calculate_textures(tumor_patches, modalities)
            non_tumor_image_texture = calculate_textures(non_tumor_patches, modalities)

        tumor_image_textures[_slice] = tumor_image_texture
        non_tumor_image_textures[_slice] = non_tumor_image_texture
    
    all_image_textures = {'tumor': tumor_image_textures, 'non_tumor_': non_tumor_image_textures}
    
    return all_image_textures


def calculate_tumor_non_tumor_texture(dataset, patient_id):
    flair_img, t2_img, t1_img, t1ce_img, mask_img = read_MRI(dataset, patient_id)
    
    stacked_image = {'t1':t1_img, 't1ce':t1ce_img, 't2':t2_img, 'flair':flair_img}
    
    # Find the slice with the largest tumor
    tumor_areas = [np.sum(mask_img[:, :, _slice]) for _slice in range(mask_img.shape[2])]
    max_tumor_slice = np.argmax(tumor_areas)

    image_textures = {}
    for modality in stacked_image.keys():
        image = stacked_image[modality]
        # Convert to uint8 for texture analysis
        sample_img_uint8 = (image / image.max() * 255).astype(np.uint8)

        # Extract the slice with the largest tumor
        slice_sample_img = sample_img_uint8[:, :, max_tumor_slice]
        slice_mask = mask_img[:, :, max_tumor_slice]

        # Separate the tumor and non-tumor regions
        tumor_region = slice_sample_img * slice_mask
        non_tumor_region = slice_sample_img * (1 - slice_mask)
        
        tumor_region = tumor_region.astype(np.uint8)
        non_tumor_region = non_tumor_region.astype(np.uint8)
            
        # Initialize dictionaries for texture properties
        textures = {'tumor': {}, 'non_tumor': {}}

        # Calculate GLCM and texture properties for each region
        for region_name, region in [('tumor', tumor_region), ('non_tumor', non_tumor_region)]:
#             print(region_name, tumor_region)
            glcm = feature.graycomatrix(region, 
                                        distances=np.linspace(1, 10, 10),
                                        angles=[0, np.pi/4, np.pi/2, 3*np.pi/4], 
                                        symmetric=True, 
                                        normed=True)
            
            # Compute texture properties
            textures[region_name]['contrast'] = feature.graycoprops(glcm, 'contrast')
            textures[region_name]['dissimilarity'] = feature.graycoprops(glcm, 'dissimilarity')
            textures[region_name]['homogeneity'] = feature.graycoprops(glcm, 'homogeneity')
            textures[region_name]['energy'] = feature.graycoprops(glcm, 'energy')
            textures[region_name]['correlation'] = feature.graycoprops(glcm, 'correlation')
            textures[region_name]['ASM'] = feature.graycoprops(glcm, 'ASM')

        image_textures[modality] = textures

    return image_textures


# Calculate the Textures of the Image 

In [47]:
def process_patient(patient_id):
    try:
        patient_id_processed = patient_id.split('GLI-')[1]
        return patient_id, calculate_overall_texture(dataset, patient_id_processed)
    except Exception as e:
        print(f"Error processing patient {patient_id}: {e}")
        return patient_id, None

In [54]:
dataset = 'Brats2023'
data_path = '../input/BraTS2023/ASNR-MICCAI-BraTS2023-GLI-Challenge-TrainingData/'
patient_ids = [f for f in listdir(data_path) if not isfile(join(data_path, f))]
# num_processes = multiprocessing.cpu_count()

image_textures = {}

# with Pool(num_processes) as pool:
#     results = pool.map(process_patient, patient_ids)
    
# image_textures = {patient_id: texture for patient_id, texture in results if texture is not None}

for _id in tqdm(patient_ids):
    patient_id = _id.split('GLI-')[1]
    image_texture = calculate_tumor_non_tumor_texture(dataset, patient_id)
    
    image_textures[_id] = image_texture

    
with open('../Results/Analysis_Results/texture/tumor_non_tumor/Texture_Analysis_tumor_vs_non_tumor.pkl', 'wb') as handle:
    pkl.dump(image_textures, handle, protocol=pkl.HIGHEST_PROTOCOL)

  0%|          | 0/1251 [00:00<?, ?it/s]

In [ ]:
restructured_data = {}

for patient_id, sequences in image_textures.items():
    for sequence, metrics in sequences.items():
        if sequence not in restructured_data:
            restructured_data[sequence] = {}
        restructured_data[sequence][patient_id] = metrics

In [ ]:
image_textures_df = pd.DataFrame.from_dict(image_textures, 
                                         orient = 'index')

for modalities in restructured_data.keys():
    columns = restructured_data[modalities][list(restructured_data[modalities].keys())[0]].keys()
    temp_df = pd.DataFrame.from_dict(restructured_data[modalities], 
                                         orient = 'index', columns = columns)
    temp_df.to_csv('../input/BraTS2023/GLI-Image_' + modalities + '_texture.csv')